In [10]:
# from gulps.synthesis_plugin import GulpsSynthesisPlugin
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.circuit.library import (
    CXGate,
    RZXGate,
    UGate,
    UnitaryGate,
    XXPlusYYGate,
    iSwapGate,
)
from qiskit.circuit.random import random_circuit
from qiskit.quantum_info import Operator, average_gate_fidelity
from qiskit.quantum_info.random import random_unitary
from qiskit.transpiler import (
    InstructionProperties,
    PassManager,
    Target,
    generate_preset_pass_manager,
)
from qiskit.transpiler.passes import Optimize1qGatesDecomposition
from tqdm import tqdm
from weylchamber import c1c2c3

from gulps.gulps_synthesis import GulpsDecomposer
from gulps.synthesis_pass import GulpsDecompositionPass
from gulps.utils.invariants import GateInvariants


### Usage as a Decomposer

In [11]:
gate_set = [
    CXGate(),
    CXGate().power(1 / 2),
    iSwapGate().power(1 / 2),
    iSwapGate().power(1 / 3),
]

costs = [1.0, 1 / 2, 1 / 2, 1 / 3]
decomposer = GulpsDecomposer(gate_set, costs, precompute_polytopes=False)

In [14]:
N = 1500
for idx in tqdm(range(N)):
    u = random_unitary(4, seed=idx)
    v = Operator(decomposer(u))
    fid = average_gate_fidelity(u, v)
    if fid < 0.9999:
        print(f"Unitary {idx} fidelity is low: {fid:.4f}")
        print(c1c2c3(u), c1c2c3(v))
        print("\n")
        break

  0%|          | 0/1500 [00:00<?, ?it/s]

[gulps.local_numerics] DEBUG: LM synthesis (attempts=1, residual=1.92e-30, success=True) in 0.089s
[gulps.local_numerics] DEBUG: LM synthesis (attempts=1, residual=9.72e-25, success=True) in 0.047s
  0%|          | 1/1500 [00:00<03:42,  6.75it/s][gulps.local_numerics] DEBUG: LM synthesis (attempts=1, residual=9.99e-32, success=True) in 0.085s
[gulps.local_numerics] DEBUG: LM synthesis (attempts=1, residual=4.24e-16, success=True) in 0.047s
  0%|          | 2/1500 [00:00<03:38,  6.86it/s][gulps.local_numerics] DEBUG: LM synthesis (attempts=1, residual=1.03e-31, success=True) in 0.081s
[gulps.local_numerics] DEBUG: LM synthesis (attempts=2, residual=3.50e-29, success=True) in 0.096s
[gulps.local_numerics] DEBUG: LM synthesis (attempts=16, residual=5.31e-03, success=False) in 0.747s
  0%|          | 3/1500 [00:01<10:11,  2.45it/s]


RuntimeError: Segment synthesis did not converge within the allotted attempts.

In [37]:
from gulps.utils.invariants import recover_local_equivalence

u = random_unitary(4, seed=9)
gi = GateInvariants.from_unitary(u, enforce_alcove=True)
vi = gi.rho_reflect

target_gate = gi.unitary
basis_gate = vi.unitary
print(average_gate_fidelity(Operator(target_gate), Operator(basis_gate)))

k1, k2, k3, k4, gphase = recover_local_equivalence(target_gate, basis_gate)
qc = QuantumCircuit(2, global_phase=gphase + np.pi)
qc.append(UnitaryGate(k1), [0])
qc.append(UnitaryGate(k2), [1])
qc.append(UnitaryGate(basis_gate), [0, 1])
qc.append(UnitaryGate(k3), [0])
qc.append(UnitaryGate(k4), [1])
print(average_gate_fidelity(Operator(target_gate), Operator(qc)))

[gulps.utils.invariants] WARNING: Possible sign difference in c parameter during local equivalence recovery.


0.3016372972991301
0.9406356299802134


In [ ]:
# %reload_ext  snakeviz
%load_ext snakeviz
import cProfile

u = random_unitary(4, seed=0)
cProfile.run("decomposer._run(u)", "profile_timings/temph.prof")